# ViT Paper Replication

Replicating **An Image is Worth 16x16 Words** (Dosovitskiy et al., 2020) on FoodVision Mini — a 3-class food image classifier (pizza / steak / sushi).

The notebook is split into two experiments:
- **From scratch** — ViT-Base/16 trained from random weights for 10 epochs
- **Fine-tuning** — pretrained ViT-B/16 backbone with frozen weights, new classification head

## 1. Setup

In [ ]:
# Uncomment to install in Colab
# !pip install torch torchvision torchinfo matplotlib Pillow tqdm requests

import os
import zipfile
import random
import requests
import torch
from torch import nn
from torchvision import transforms
from torchvision.models import vit_b_16, ViT_B_16_Weights
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

from vit.data import create_dataloaders
from vit.model import PatchEmbedding, MultiHeadSelfAttentionBlock, ViT
from vit.engine import train
from vit.utils import save_model
from predict import pred_and_plot_image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
torch.manual_seed(42)
random.seed(42)

## 2. Data Exploration

FoodVision Mini is a 3-class subset of the Food101 dataset (~75 train / ~25 test images per class). Small enough to overfit easily — which makes it a useful stress test for data efficiency.

In [ ]:
DATA_URL = 'https://github.com/mrdbourke/pytorch-deep-learning/releases/download/misc/pizza_steak_sushi.zip'

def download_data(source: str, destination: str, remove_source: bool = True) -> Path:
    data_path = Path('data/')
    image_path = data_path / destination
    if image_path.is_dir():
        print(f'[INFO] {image_path} already exists, skipping download.')
    else:
        print(f'[INFO] Creating {image_path}...')
        image_path.mkdir(parents=True, exist_ok=True)
        target_file = Path(source).name
        with open(data_path / target_file, 'wb') as f:
            print(f'[INFO] Downloading {target_file}...')
            response = requests.get(source)
            response.raise_for_status()
            f.write(response.content)
        with zipfile.ZipFile(data_path / target_file, 'r') as zip_ref:
            print(f'[INFO] Unzipping {target_file}...')
            zip_ref.extractall(image_path)
        if remove_source:
            os.remove(data_path / target_file)
    return image_path

image_path = download_data(source=DATA_URL, destination='pizza_steak_sushi')
print(f'Data path: {image_path}')

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for i, class_name in enumerate(['pizza', 'steak', 'sushi']):
    class_dir = image_path / 'train' / class_name
    images = list(class_dir.glob('*.jpg'))[:4]
    for j, img_path in enumerate(images):
        axes[i, j].imshow(mpimg.imread(img_path))
        axes[i, j].set_title(class_name, fontsize=9)
        axes[i, j].axis('off')
plt.suptitle('FoodVision Mini — Sample Images', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
class_names = ['pizza', 'steak', 'sushi']
train_counts = [len(list((image_path / 'train' / c).glob('*.jpg'))) for c in class_names]
test_counts  = [len(list((image_path / 'test'  / c).glob('*.jpg'))) for c in class_names]

x = range(len(class_names))
width = 0.35
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([i - width/2 for i in x], train_counts, width, label='train', color='steelblue')
ax.bar([i + width/2 for i in x], test_counts,  width, label='test',  color='salmon')
ax.set_xticks(list(x)); ax.set_xticklabels(class_names)
ax.set_title('Samples per class'); ax.set_ylabel('Count'); ax.legend()
plt.tight_layout(); plt.show()

## 3. Patch Embedding Walkthrough

A `Conv2d` with `kernel_size=patch_size` and `stride=patch_size` maps each non-overlapping 16×16 pixel patch to a 768-dimensional embedding vector. A 224×224 image produces exactly 14×14 = 196 patches. A learnable class token is prepended (index 0), giving a sequence of 197 tokens.

The positional embedding is a learnable 1D table of 197 vectors — the model learns spatial structure entirely from data.

In [ ]:
img = torch.randn(1, 3, 224, 224)
print(f'Input:            {img.shape}')  # [1, 3, 224, 224]

patcher = PatchEmbedding()
patches = patcher(img)
print(f'After PatchEmbed: {patches.shape}')  # [1, 196, 768]

class_token = nn.Parameter(torch.randn(1, 1, 768))
x = torch.cat([class_token.expand(1, -1, -1), patches], dim=1)
print(f'+ class token:    {x.shape}')  # [1, 197, 768]

pos_embed = nn.Parameter(torch.randn(1, 197, 768))
x = x + pos_embed
print(f'+ pos embedding:  {x.shape}')  # [1, 197, 768]

In [ ]:
# Visualise the 196 patches extracted from a sample image
from PIL import Image as PILImage
sample_img_path = next((image_path / 'train' / 'pizza').glob('*.jpg'))
img_pil = PILImage.open(sample_img_path).convert('RGB')
img_tensor = transforms.Compose([
    transforms.Resize((224, 224), antialias=True),
    transforms.ToTensor(),
])(img_pil)

fig, axes = plt.subplots(14, 14, figsize=(8, 8))
patch_imgs = img_tensor.unfold(1, 16, 16).unfold(2, 16, 16)  # [3, 14, 14, 16, 16]
for row in range(14):
    for col in range(14):
        patch = patch_imgs[:, row, col, :, :].permute(1, 2, 0).numpy()
        axes[row, col].imshow(patch.clip(0, 1))
        axes[row, col].axis('off')
plt.suptitle('196 patches (16x16 each)', fontsize=12)
plt.tight_layout(); plt.show()

## 4. ViT Architecture from Scratch

The full architecture stacks 12 `TransformerEncoderBlock`s, each comprising:
- **MSA**: LayerNorm → MultiheadAttention (12 heads) → residual
- **MLP**: LayerNorm → Linear(768→3072) → GELU → Dropout → Linear(3072→768) → residual

The final representation is taken from the class token `x[:, 0]` and passed through a `LayerNorm + Linear` head.

In [ ]:
try:
    from torchinfo import summary
    TORCHINFO = True
except ImportError:
    TORCHINFO = False
    print('pip install torchinfo for layer summaries')

In [ ]:
if TORCHINFO:
    print('--- PatchEmbedding ---')
    summary(PatchEmbedding(), input_size=(1, 3, 224, 224),
            col_names=['input_size', 'output_size', 'num_params'])

In [ ]:
if TORCHINFO:
    print('--- MultiHeadSelfAttentionBlock ---')
    summary(MultiHeadSelfAttentionBlock(), input_size=(1, 197, 768))

In [ ]:
if TORCHINFO:
    print('--- Full ViT-Base/16 (86M params) ---')
    summary(ViT(), input_size=(1, 3, 224, 224))

## 5. Training from Scratch

ViT requires significantly more data and training than CNNs to learn visual features from scratch — the paper trains on JFT-300M (300 million images). With only ~225 training images here, we expect modest accuracy. The goal is to verify the implementation runs and tracks learning, not to match paper results.

In [ ]:
custom_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
train_dl, test_dl, class_names = create_dataloaders(
    train_dir=image_path / 'train',
    test_dir=image_path / 'test',
    transform=custom_transform,
    batch_size=32,
)

In [ ]:
custom_vit = ViT(num_classes=len(class_names)).to(device)
optimizer = torch.optim.Adam(custom_vit.parameters(), lr=3e-3, weight_decay=0.1)
loss_fn = nn.CrossEntropyLoss()

custom_results = train(
    model=custom_vit,
    train_dataloader=train_dl,
    test_dataloader=test_dl,
    optimizer=optimizer,
    loss_fn=loss_fn,
    epochs=10,
    device=device,
)

In [ ]:
save_model(custom_vit, target_dir='models', model_name='custom_vit.pth')

epochs = range(len(custom_results['train_loss']))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))
ax1.plot(epochs, custom_results['train_loss'], label='train_loss')
ax1.plot(epochs, custom_results['test_loss'], label='test_loss')
ax1.set_title('Loss'); ax1.set_xlabel('Epochs'); ax1.legend()
ax2.plot(epochs, custom_results['train_acc'], label='train_accuracy')
ax2.plot(epochs, custom_results['test_acc'], label='test_accuracy')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epochs'); ax2.legend()
plt.tight_layout(); plt.show()

## 6. Pretrained ViT-B/16 Fine-tuning

Loading ImageNet-pretrained weights gives the model rich visual features for free. We freeze the entire backbone (85.8M params) and only train the replacement classification head (2,307 params). This is the standard recipe when labelled data is scarce.

In [ ]:
weights = ViT_B_16_Weights.DEFAULT
pretrained_vit = vit_b_16(weights=weights)
for param in pretrained_vit.parameters():
    param.requires_grad = False
pretrained_vit.heads = nn.Linear(in_features=768, out_features=len(class_names))
pretrained_vit = pretrained_vit.to(device)
trainable = sum(p.numel() for p in pretrained_vit.parameters() if p.requires_grad)
total     = sum(p.numel() for p in pretrained_vit.parameters())
print(f'Trainable params: {trainable:,} / {total:,}')

In [ ]:
auto_transforms = weights.transforms()
train_dl_pt, test_dl_pt, _ = create_dataloaders(
    train_dir=image_path / 'train',
    test_dir=image_path / 'test',
    transform=auto_transforms,
    batch_size=32,
)
optimizer_pt = torch.optim.Adam(
    params=filter(lambda p: p.requires_grad, pretrained_vit.parameters()),
    lr=1e-3,
)
pretrained_results = train(
    model=pretrained_vit,
    train_dataloader=train_dl_pt,
    test_dataloader=test_dl_pt,
    optimizer=optimizer_pt,
    loss_fn=loss_fn,
    epochs=5,
    device=device,
)

In [ ]:
save_model(pretrained_vit, target_dir='models', model_name='pretrained_vit.pth')

epochs = range(len(pretrained_results['train_loss']))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))
ax1.plot(epochs, pretrained_results['train_loss'], label='train_loss')
ax1.plot(epochs, pretrained_results['test_loss'], label='test_loss')
ax1.set_title('Loss'); ax1.set_xlabel('Epochs'); ax1.legend()
ax2.plot(epochs, pretrained_results['train_acc'], label='train_accuracy')
ax2.plot(epochs, pretrained_results['test_acc'], label='test_accuracy')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epochs'); ax2.legend()
plt.tight_layout(); plt.show()

## 7. Results Comparison

In [ ]:
header = f"{'Model':<30} {'Test Acc':>10} {'Test Loss':>12}"
print(header)
print('-' * 55)
print(f"{'ViT from scratch (10 ep)':<30} "
      f"{custom_results['test_acc'][-1]:>10.3f} "
      f"{custom_results['test_loss'][-1]:>12.4f}")
print(f"{'Pretrained ViT-B/16 (5 ep)':<30} "
      f"{pretrained_results['test_acc'][-1]:>10.3f} "
      f"{pretrained_results['test_loss'][-1]:>12.4f}")

**Takeaway:** The pretrained model converges in 5 epochs to accuracy the from-scratch model cannot reach in 10 — consistent with the paper's finding that ViT is data-hungry: without large-scale pre-training, inductive biases built into CNNs give them the edge on small datasets. Transfer learning collapses the data gap by reusing the ImageNet-learned patch representations.

## 8. Predictions on Test Images

Using the pretrained fine-tuned model for inference — it has the strongest test accuracy.

In [ ]:
test_images = (
    list((image_path / 'test' / 'pizza').glob('*.jpg'))[:2] +
    list((image_path / 'test' / 'steak').glob('*.jpg'))[:2] +
    list((image_path / 'test' / 'sushi').glob('*.jpg'))[:2]
)
random.shuffle(test_images)

for img_path in test_images:
    pred_and_plot_image(
        model=pretrained_vit,
        class_names=class_names,
        image_path=str(img_path),
        transform=auto_transforms,
        device=device,
    )
plt.show()